## SQLite

SQLite is a self-contained, file-based SQL database.

SQLite is not directly comparable to client/server SQL database engines such as Oracle, PostgreSQL or MySQL, it is intended to provide local data storage for individual applications and devices.

SQLite comes bundled with Python and can be used in any Python applications without having to install any additional software.

In [6]:
import sqlite3
from sqlite3 import Error

### Creating a connection to a SQLite database

The sqlite3.connect() function returns a Connection object that can be used to interact with the SQLite database (that ultimately resides in a file). The database (file) will be created automatically if it does not exists.

In [8]:
def create_connection(db_file):
    """ Create a database connection to a SQLite database
    Parameters
    ----------
    db_file: database file
    Returns
    -------
    Connection object or None
    """
    conn = None
    try:
        conn = sqlite3.connect(db_file)
        # print(sqlite3.version)
        return conn
    except Error as e:
        print(e)

    return conn

### Creating tables

In order to execute SQL statements like CREATE TABLE, you need to first create a Cursor object, by calling the cursor() method of the Connection object, and then pass the SQL statement to the execute() method of the Cursor object.

In [9]:
def create_table(conn, create_table_sql):
    """ Create a table from a CREATE TABLE SQL statement
    Parameters
    ----------
    conn: Connection object
    create_table_sql: a CREATE TABLE statement
    """
    try:
        c = conn.cursor()
        c.execute(create_table_sql)
    except Error as e:
        print(e)

### Inserting data

To use a variable in a SQL statement, write a question mark (?) as a placeholder for each argument, and substitute the actual values by providing them as a tuple of values to the cursor’s execute() method.

In SQLite, a column with type INTEGER PRIMARY KEY is an alias for the ROWID, a permanent unique identifier for a row. On an INSERT, if the INTEGER PRIMARY KEY column is not explicitly given a value, then it will be filled automatically with an unused integer, usually one more than the largest ROWID currently in use.

In [10]:
def insert_new_journal(conn, journal):
    """
    Insert a new journal into the journal table
    Parameters
    ----------
    conn: Connection object
    journal: Values to bind to placeholders in sql statement.
    Returns
    -------
    the row id of the inserted row.
    """
    sql = ''' INSERT INTO journal(journal_name, publisher)
              VALUES(?,?) '''
    cur = conn.cursor()
    cur.execute(sql, journal)
    conn.commit()
    return cur.lastrowid

### Querying Data

The fetchall() method of the Cursor object fetches all the rows of a query result. It returns all the rows as a list of tuples. An empty list is returned if there is no record to fetch. 

In [25]:
def select_journals_by_publisher(conn, publisher):
    """
    Query journals by publisher
    Parameters
    ----------
    conn: Connection object
    publisher: Value to bind to placeholder in sql statement.
    """
    cur = conn.cursor()

    # For the question marks style, parameters must be a sequence e.g. a tuple
    cur.execute("SELECT * FROM journal WHERE publisher=?", (publisher,))

    # Never use string operations to assemble queries, as they are vulnerable to SQL injection attacks
    # sql = "SELECT * FROM journal WHERE publisher='%s'" % publisher
    # print(sql)
    # cur.execute(sql)

    rows = cur.fetchall()

    for row in rows:
        print(row)

In [18]:
# Create a database connection (the database will be created 
# automatically if it does not exists).

database = "./journal_articles.db"

conn = create_connection(database)
if conn is None:
   print("Cannot create the database connection.") 

In [19]:
# Create tables

sql_create_journal_table = """
CREATE TABLE IF NOT EXISTS journal (
    journal_id integer PRIMARY KEY,
    journal_name text NOT NULL,
    publisher text DEFAULT 'Springer Nature' NOT NULL
    ); 
"""

if conn is not None:
    # create journal table
    create_table(conn, sql_create_journal_table)
else:
    print("A valid connection object is required.")

In [20]:
# Insert data

if conn is not None:
    journal = ('Neurocomputing', 'Elsevier');
    journal_id = insert_new_journal(conn, journal)
    journal = ('IEEE Access', 'IEEE');
    journal_id = insert_new_journal(conn, journal)
else:
    print("A valid connection object is required.")

In [27]:
# Querying data

if conn is not None:
    publisher = 'Elsevier'
    # publisher = "' OR TRUE; --"
    select_journals_by_publisher(conn, publisher)

else:
    print("A valid connection object is required.")

(1, 'Neurocomputing', 'Elsevier')


In [28]:
if conn:
    conn.close()

In [29]:
%%shell
jupyter nbconvert --to html /content/SQLite.ipynb

[NbConvertApp] Converting notebook /content/SQLite.ipynb to html
[NbConvertApp] Writing 288730 bytes to /content/SQLite.html
